# C# and SQLite

```{contents}
:local:
:depth: 2
```

Use a saved .NET console project for this workflow. It creates files and uses a provider package, so complete it in VS Code rather than a browser code cell. The project uses classes, nullable results, collections, exceptions, and `using` declarations. Review those core chapters before starting.


## Set Up and Run

```{index} Set Up and Run
```


Download or clone the book repository and open `demos/_extras/bookstore_inventory/`. Its project file pins the SQLite provider version used by the reference implementation.

```sh
cd demos/_extras/bookstore_inventory
dotnet restore
dotnet run
dotnet run -- --check
```

The first restore requires internet access. No database server or account is needed. Run from the project directory so the relative database path is predictable.

The ordinary run creates `bookstore.db` and lists two seeded titles. Later runs preserve data; seed IDs are inserted only if absent. `--check` uses a new in-memory database, leaving your saved inventory untouched. To reset practice data, stop the program and delete only your local `bookstore.db`, then run again.

- [Project file](../../../demos/_extras/bookstore_inventory/BookstoreInventory.csproj)
- [Complete reference source](../../../demos/_extras/bookstore_inventory/Program.cs)
- {download}`Project instructions <../../../demos/_extras/bookstore_inventory/README.md>`


## Connections, Commands, and Disposal

```{index} Connections, Commands, and Disposal
```


A `SqliteConnection` opens the local database. A command holds SQL text and parameters. A reader steps through result rows. `using` disposes each resource when its scope ends, including when an exception leaves the scope. This `using` declaration manages a resource; it has a different role from a namespace import.

The reference project follows this order: open a connection, enable foreign keys, create tables, seed missing sample records, construct an `Inventory`, and request operations. Inspect the setup before the class definitions in `Program.cs`.


The following complete `Program.cs` demonstrates one connection, setup command, parameter, and reader. Run it in a separate console project with the same SQLite package as the supplied project. Its in-memory database is recreated each run. Expected output: `C# for Business`.

```csharp
using Microsoft.Data.Sqlite;

using var connection = new SqliteConnection("Data Source=:memory:");
connection.Open();
using (var setup = connection.CreateCommand())
{
    setup.CommandText = """
        CREATE TABLE Books (BookId INTEGER PRIMARY KEY, Title TEXT NOT NULL);
        INSERT INTO Books VALUES (1, 'C# for Business');
        """;
    setup.ExecuteNonQuery();
}
using var query = connection.CreateCommand();
query.CommandText = "SELECT Title FROM Books WHERE BookId = @id;";
query.Parameters.AddWithValue("@id", 1);
using var reader = query.ExecuteReader();
while (reader.Read())
    Console.WriteLine(reader.GetString(0));
```


## Map Rows to Objects

```{index} Map Rows to Objects
```


`Book` groups an ID, title, price in cents, and stock. `Inventory.List` uses an explicit column order and reads each row into a new `Book`. `Find` returns either one book or `null`; missing data is an expected result, not automatically an exception.

Read `ReadBook` and match every reader position to its selected column. Predict what could go wrong if a query changed column order without updating the mapper. Keep each command's result shape explicit.


## Supply Parameters and Check Outcomes

```{index} Supply Parameters and Check Outcomes
```


`Add` passes the title, price, and stock as values. A title such as `Manager's SQL Guide` needs no manual quote replacement. `SetStock` uses an ID condition and reports whether exactly one row changed. Reject invalid quantities before sending the command, and keep database constraints as a second boundary.

In the source, compare `Find` with `SetStock`: both use an ID parameter, but one reads a row and the other checks the affected-row count. Explain the expected result for an unknown ID in each operation.


## Transactions for Related Changes

```{index} Transactions for Related Changes
```


The optional `Sell` operation begins a transaction, reduces stock only if enough remains, inserts a sale, and commits. If the book is missing or stock is insufficient, it rolls back and returns `false`. If an unexpected error occurs before commit, disposal rolls back the uncommitted transaction.

Every command participating in the transaction sets its `Transaction` property. Updating stock and inserting the sale independently could leave only one change saved after a failure. Atomicity means these changes are committed together.


## Verify Before Extending

```{index} Verify Before Extending
```


`dotnet run -- --check` checks seed data, missing IDs, apostrophes in titles, updates, successful sales, overselling, and invalid inputs. Read the assertions before editing. Then run twice without `--check` and explain why the stored inventory persists while the check database starts fresh each time.

Keep real customer data and production credentials out of this exercise. For a semester project, document the schema, initialization, reset procedure, and tested failure cases alongside the program.


```{rubric} Footnotes
```
[^1]: Use the supplied local teaching database, rather than a production database.
